# Alchemy GeomML + TDA — Quickstart (v33+)

Полный пайплайн: загрузка данных → обучение → оценка → AutoML.

**Датасет:** Alchemy v20191129 (202,579 молекул, 12 квантово-механических свойств)

**Модели:** FCNN, SchNet, EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA, **EGNN Tensor** (часть B)

## 1. Установка зависимостей

Если запуск локально (не в Kaggle/Colab):

In [ ]:
!pip install torch torch-geometric gudhi rdkit egnn-pytorch pandas matplotlib seaborn pyyaml scikit-learn

## 2. Клонирование репозитория и загрузка данных

In [ ]:
import os
if not os.path.exists('alchemy-geom-tda'):
    !git clone https://github.com/ThomasMoore25/alchemy-geom-tda.git
%cd alchemy-geom-tda
!git log --oneline -1

In [ ]:
# Скачать датасет Alchemy (136 МБ zip, ~600 МБ распакованный)
# Подробный вывод — видно прогресс скачивания и проверки SHA256
!python data/download_alchemy.py

## 3. Проверка окружения

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
import torch_geometric
import egnn_pytorch
import rdkit
import gudhi

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'torch-geometric: {torch_geometric.__version__}')
print(f'rdkit: {rdkit.__version__}')
print(f'gudhi: {gudhi.__version__}')

import src
print(f'\nalchemy-geom-tda version: {src.__version__}')

## 4. Smoke-тест (5 минут на CPU, проверка установки)

Обучаем EGNN на 100 молекулах, 2 эпохи. Должен быть виден убывающий loss.

In [ ]:
import sys, os
sys.path.insert(0, 'src')
os.makedirs('results/smoke', exist_ok=True)

sys.argv = ['train.py',
    '--model', 'egnn',
    '--target', 'all',
    '--epochs', '2',
    '--max_train', '100',
    '--max_val', '20',
    '--max_test', '20',
    '--batch_size', '32',
    '--device', 'cpu',
    '--output_dir', 'results/smoke',
]

from train import main as train_main
train_main()

## 5. Полное обучение одной модели (GPU, ~1-2 часа)

EGNN на полном датасете с быстрым EarlyStopping.

**Параметры для быстрой сходимости:**
- `--epochs 9999` (EarlyStopping остановит раньше)
- `--patience 10` (вместо 15)
- `--es_mode or` (стоп при первой плохой метрике)
- `--batch_size 1024` (если GPU позволяет)

In [ ]:
# Раскомментировать для полного запуска (нужен GPU)

# sys.argv = ['train.py',
#     '--model', 'egnn',
#     '--target', 'all',
#     '--epochs', '9999',
#     '--batch_size', '1024',
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--lr_patience', '5',
#     '--device', 'cuda',
#     '--patience', '10',
#     '--es_mode', 'or',
#     '--num_workers', '4',
#     '--output_dir', 'results/experiments/batch_size_1024',
# ]
# from train import main as train_main
# train_main()

## 6. EGNN Tensor — часть B (вектор μ + тензор α)

**EGNN Tensor** (v33.0+, программа максимума) — физически корректная модель:
- Предсказывает **вектор** дипольного момента μ ∈ R³ (а не скаляр |μ|)
- Предсказывает **тензор** поляризуемости α ∈ R^(3×3) (а не скаляр tr(α)/3)
- Через частичные заряды атомов: q_i = MLP(h_atom)
- μ = Σᵢ qᵢ · (rᵢ − COM), α = Σᵢ qᵢ · (rᵢ − COM) ⊗ (rᵢ − COM)

**Чем отличается от обычного EGNN:**
- Обычный EGNN: предсказывает только скаляры |μ| и α_iso
- EGNN Tensor: предсказывает полный вектор μ и полный тензор α
- Это даёт больше информации: направление μ, анизотропию α
- E(3)-эквивариантность 1-го и 2-го ранга (l=1, l=2)

In [ ]:
# Раскомментировать для запуска EGNN Tensor (часть B, ~1-2 часа на GPU)

# sys.argv = ['train.py',
#     '--model', 'egnn_tensor',         # часть B: вектор μ + тензор α
#     '--target', 'all',
#     '--epochs', '9999',
#     '--batch_size', '1024',
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--lr_patience', '5',
#     '--device', 'cuda',
#     '--patience', '10',
#     '--es_mode', 'or',
#     '--predict_tensor_alpha',         # предсказывать полный тензор α
#     '--num_workers', '4',
#     '--output_dir', 'results/experiments/batch_size_1024',
# ]
# from train import main as train_main
# train_main()

## 7. Обучение всех 7 моделей (run_all)

In [ ]:
# Раскомментировать для полного запуска (3-6 часов на GPU)

# sys.argv = ['run_all.py',
#     '--models', 'all',
#     '--epochs', '9999',
#     '--batch_size', '1024',
#     '--device', 'cuda',
#     '--patience', '10',
#     '--es_mode', 'or',
#     '--num_workers', '4',
#     '--multi_gpu',  # если 2+ GPU
# ]
# from run_all import main as run_all_main
# run_all_main()

## 8. Графики обучения

Кривые train/val loss и MAE по эпохам + parity plots.

In [ ]:
import sys
sys.path.insert(0, 'src')
from plot import plot_training_history, plot_parity, compare_histories
import glob, os

# Укажите путь к history CSV (замените на актуальный)
csv_path = 'results/smoke/history_egnn_all_20260717_091245.csv'  # smoke-test
# csv_path = 'results/experiments/batch_size_1024/history_egnn_all_20260712_160620.csv'  # полный

if os.path.exists(csv_path):
    plot_training_history(csv_path, show=True)
    plot_parity(csv_path, show=True)
else:
    print(f'CSV не найден: {csv_path}')
    print('Доступные history CSV:')
    for f in sorted(glob.glob('results/**/history_*.csv', recursive=True)):
        print(f'  {f}')

## 9. Robustness evaluation

Оценка устойчивости обученной модели к шуму в координатах (σ ∈ {0.0, 0.05, 0.10, 0.15}).

In [ ]:
# Раскомментировать после обучения модели

# sys.argv = ['eval_robustness.py',
#     '--model', 'egnn', '--target', 'all',
#     '--checkpoint', 'checkpoints/egnn_all_best.pt',
#     '--target_stats', 'results/experiments/batch_size_1024/target_stats_egnn_all.json',
#     '--noise_sigma', '0.0,0.05,0.10,0.15',
#     '--device', 'cuda',
# ]
# from eval_robustness import main as eval_main
# eval_main()

## 10. AutoML — автоматический выбор архитектуры (программа максимум)

Извлекает геометрические priors из TDA и рекомендует оптимальную архитектуру.

Примечание: файл переименован из `select.py` в `run.py` (v33.2), чтобы избежать конфликта со стандартным модулем Python `select`.

In [ ]:
import sys
sys.path.insert(0, 'src')
sys.path.insert(0, 'src/automl')

sys.argv = ['run.py',
    '--data_dir', 'data/alchemy',
    '--n_molecules', '30',
    '--threshold', '0.95',
    '--output_json', 'results/automl/recommendation.json',
]

# ВАЖНО: импорт из run, а не из select (конфликт со stdlib)
from run import main as automl_main
automl_main()

## 11. Просмотр отчёта AutoML

In [ ]:
import json
from pathlib import Path

report_path = 'results/automl/recommendation.json'
if Path(report_path).exists():
    with open(report_path) as f:
        report = json.load(f)
    print('=== AutoML Report ===')
    print(f'Timestamp: {report["timestamp"]}')
    print(f'\nPriors (n={report["priors"]["n_molecules_success"]} molecules):')
    for k, v in report['priors'].items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
    print(f'\nRecommendation:')
    for k, v in report['recommendation'].items():
        print(f'  {k}: {v}')
else:
    print(f'Отчёт не найден: {report_path}')

## 12. Тесты (краткий вывод)

In [ ]:
# Краткий вывод: -q (quiet), --tb=line (одна строка на ошибку)
!python -m pytest -q --tb=line 2>&1 | tail -5

## Что дальше

1. **Полное обучение на GPU:** раскомментируйте ячейки 5-7, установите `--device cuda`.
2. **EGNN Tensor (часть B):** ячейка 6 — физически корректная модель с векторным μ и тензорным α.
3. **Robustness eval:** после обучения прогоните ячейку 9 на каждой модели.
4. **AutoML с quick_train:** добавьте `--quick_train --candidates fcnn,schnet,egnn,egnn_tda`.

См. `QUICKSTART.md` для краткой справки по CLI.